In [22]:
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import Food101
from torchvision.models import vit_b_16, ViT_B_16_Weights

datadir = "datasets/torchvision/food101"
train_result_dir = "hf_models/finetuned_food101"
device = "cuda"
batch_size = 32

weights = ViT_B_16_Weights.DEFAULT
image_transform = weights.transforms()
device = "cuda"

In [23]:
train_dataset = Food101(root=datadir, split="train", transform=image_transform, download=True)
test_dataset = Food101(root=datadir, split="test", transform=image_transform, download=True)

In [24]:
from torch.utils.data import DataLoader, Dataset

train_loaders = DataLoader(train_dataset, batch_size, shuffle=True, pin_memory=True)
test_loaders = DataLoader(test_dataset, batch_size, pin_memory=True)

model = vit_b_16(weights=weights)

head_input_dim:int = model.heads.head.in_features
model.heads.head = nn.Linear(
    head_input_dim, 
    len(train_dataset.classes)
    )

In [25]:
from typing import TypedDict
from torch import Tensor


class Dictformat(TypedDict):
    pixel_values: Tensor
    labels: int

class DictDataset(Dataset):
    def __init__(self, raw_dataset):
        self.raw_dataset = raw_dataset
    
    def __len__(self):
        return len(self.raw_dataset)
    
    def __getitem__(self, index) -> Dictformat:
        image, label = self.raw_dataset[index]
        return {"pixel_values":image, "labels":label}

train_dataset_hf = DictDataset(train_dataset)
test_dataset_hf = DictDataset(test_dataset)

In [26]:
from typing import Optional
import numpy as np

class VitReturnFormat(TypedDict):
    loss:Optional[float]
    logits:Tensor

class VitFineTuneModel(nn.Module):
    def __init__(self, vit_model):
        super().__init__()
        self.vit_model = vit_model
        self.loss_fn = nn.CrossEntropyLoss()
    
    def forward(self, pixel_values:Tensor, labels:Optional[Tensor]=None) ->VitReturnFormat:
        logit = self.vit_model(pixel_values)
        
        if labels is None:
            return {"loss":None, "logits":logit} #inference?
        
        #train?
        loss = self.loss_fn(logit, labels)
        return {"loss":loss, "logits":logit}

def compute_metrics(pred):
    logits, labels = pred
    max_preds = np.argmax(logits, axis=-1)
    return {"accuracy": (max_preds == labels).mean()}


hf_model = VitFineTuneModel(model).to(device)

In [29]:
from transformers import Trainer, TrainingArguments

train_args = TrainingArguments(
    output_dir=train_result_dir, eval_strategy="epoch", save_strategy="epoch",
    per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
    num_train_epochs=3, learning_rate=5e-5, logging_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="accuracy", remove_unused_columns=False
)

trainer = Trainer(hf_model, train_args, train_dataset=train_dataset_hf, eval_dataset=test_dataset_hf, compute_metrics=compute_metrics)

In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.943292,0.629112,0.829743
2,0.426092,0.536560,0.851485
3,0.141764,0.522577,0.859248


TrainOutput(global_step=7104, training_loss=0.5037158373239878, metrics={'train_runtime': 2544.073, 'train_samples_per_second': 89.325, 'train_steps_per_second': 2.792, 'total_flos': 0.0, 'train_loss': 0.5037158373239878, 'epoch': 3.0})

In [ ]:
trainer.evaluate()